# E: Pipeline demo (detector -> explain -> guidebook RAG -> Llama rewrite -> checks)

Runs the real pieces together and prints every step for each sentence.

| Step | Code | Owner | State |
|---|---|---|---|
| 1 Detector | `detector.detect` (XLM-R from Drive) | Iroda | real |
| 2 Explainer | `detector.explain` (Captum) | Iroda | real |
| 3 RAG | `rag.retrieve` over `data/guidebook.md` | Yasa, Amen | real |
| 4 Rewriter | `rewriter.rewrite` (Llama 3.1 8B 4-bit, Qwen fallback) | Madina | real |
| 5 Fact check | `pipeline.fact_check` (numbers, names; NLI later) | integration | real (simple) |
| 6 Reviewers A/B | `reviewers.review` | Madina | **fake: always "fair"** until her prompts land |
| 7 Evaluator | `pipeline.evaluate` | integration | real rules |

Needs: T4 GPU; the trained detector on Drive; the guidebook from PR #6 on `main`;
`HF_TOKEN` with access to Llama 3.1 (otherwise the rewriter falls back to Qwen2.5-7B, which needs no approval).

In [1]:
# === TEAM SETUP CELL: run first. Same on Colab, Kaggle and local Jupyter. ===
# Do not edit this cell in your notebook; change src/ or requirements.txt in the repo instead.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:  # autoreload is broken on some Colab images (Python 3.13: no 'imp' module)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload not available: restart the runtime after editing src/")

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

Mounted at /content/drive
autoreload not available: restart the runtime after editing src/
repo: /content/debias | artifacts: /content/drive/MyDrive/debias
runtime=colab  python=3.13.16  gpu=Tesla T4  commit=a080f2c
  torch                  2.14.1+cu130
  transformers           5.23.0
  datasets               5.2.0
  accelerate             1.15.0
  bitsandbytes           0.50.2
  sentence-transformers  6.3.0
  faiss-cpu              1.16.0
  captum                 0.9.0
  scikit-learn           1.6.1
  pandas                 2.2.3
  numpy                  2.4.6


## 1. Point to the trained detector and check the guidebook

In [2]:
from src import detector, rag, rewriter, reviewers, pipeline

DETECTOR_DIR = ARTIFACTS / "detector" / "xlm-roberta-base_seed44" / "model"  # best seed (by validation)
assert DETECTOR_DIR.exists(), f"No trained detector at {DETECTOR_DIR}; run A_detector first"
detector.set_model_path(DETECTOR_DIR)
print("thresholds:", detector.thresholds())

entries = rag.load_entries(config.GUIDEBOOK_PATH)
print(len(entries), "guidebook entries:", [rag.entry_id(e) for e in entries])
if len(entries) < 5:
    print("WARNING: the guidebook is nearly empty. Merge Amen's PR #6, then rerun the setup cell.")

thresholds: {'biased': 0.58, 'uncertain_low': 0.12108474969863892, 'val_macro_f1': 0.7201166024695436}
25 guidebook entries: ['R01', 'R02', 'R03', 'R04', 'R05', 'R06', 'R07', 'R08', 'R09', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15', 'R16', 'R17', 'R18', 'R19', 'R20', 'R21', 'R22', 'R23', 'R24', 'R25']


## 2. Load all models once (and time it)

In [3]:
import time, torch

def timed(label, fn):
    start = time.perf_counter(); out = fn(); print(f"{label:28s} {time.perf_counter() - start:6.1f} s"); return out

timed("detector (XLM-R)", lambda: detector.detect("warm up"))
timed("RAG (MiniLM + FAISS)", lambda: rag.retrieve("warm up"))
_, _, model_id = timed("rewriter (4-bit LLM)", rewriter.get_model)
print("rewriter model:", model_id)
if torch.cuda.is_available():
    print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB of {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

detector (XLM-R)               33.3 s


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (17 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[rewriter] meta-llama/Llama-3.1-8B-Instruct failed to load, using Qwen/Qwen2.5-7B-Instruct
rewriter (4-bit LLM)          254.4 s
rewriter model: Qwen/Qwen2.5-7B-Instruct
GPU memory in use: 7.2 GB of 15.6 GB


## 3. One sentence, step by step

`force=True` pushes a sentence through every step even if the detector says "unbiased" or "uncertain"
(the real detector verdict is still shown). Use it only to look at the chain, never for evaluation numbers.

In [4]:
def rule_ids(trace):
    return [r.split()[1] for r in trace.get("rules", [])]

def show(result, trace):
    d = trace["detect"]
    forced = trace["forced"] and d["status"] != "biased"
    print("INPUT:", trace["input"])
    print(f"1 DETECTOR   status={d['status']}  score={d['score']:.2f}  label={d['label']}" + ("   (forced through)" if forced else ""))
    if "spans" not in trace:
        print("  -> stops here:", result["decision"], "|", result.get("reason", ""))
        return
    print(f"2 EXPLAIN    flagged words: {trace['spans']}")
    print(f"3 RAG        query={trace['query']!r}  rules={rule_ids(trace)}")
    for rd in trace["rounds"]:
        rw, f = rd["rewrite"], rd["fact_check"]
        print(f"4 REWRITE    round {rd['round']}: {rw['revised']}")
        for c in rw["changes"]:
            print(f'               "{c["original"]}" -> "{c["revised"]}"   [{c["reason"]}]')
        if not rw["changes"]:
            print("               (no changes)")
        print(f"5 FACT CHECK ok={f['ok']}  new numbers={f['new_numbers']}  new names={f['new_names']}")
        for p, r in rd["reviews"].items():
            print(f"6 REVIEW     {p:12s} fair={r['fair']}  issues={r['issues']}")
        print(f"7 EVALUATOR  {rd['decision']}")
    print("FINAL:", result["final"])
    print(result["change_note"])
    print("seconds:", trace["seconds"])

result, trace = pipeline.run_detailed(
    "Hamas terrorists butchered innocent families in a barbaric massacre.", force=True)
show(result, trace)

INPUT: Hamas terrorists butchered innocent families in a barbaric massacre.
1 DETECTOR   status=biased  score=0.99  label=against_SIDE_B
2 EXPLAIN    flagged words: ['terrorists', 'innocent', 'barbaric', 'butchered', 'families']
3 RAG        query='terrorists innocent barbaric butchered families'  rules=['R01', 'R04', 'R18', 'R15']
4 REWRITE    round 1: Hamas attacked families in an attack that killed several people.
               "Hamas terrorists butchered innocent families in a barbaric massacre." -> "Hamas attacked families in an attack that killed several people."   [R01: actor not stated in the text, R04: dehumanizing language, R18: terminology]
5 FACT CHECK ok=True  new numbers=[]  new names=[]
6 REVIEW     palestinian  fair=True  issues=[]
6 REVIEW     israeli      fair=True  issues=[]
7 EVALUATOR  approve
FINAL: Hamas attacked families in an attack that killed several people.
Changes made to this answer:
- "Hamas terrorists butchered innocent families in a barbaric massacre."

In [5]:
# Same kind of wording against the other side: the chain should treat it the same way.
result, trace = pipeline.run_detailed(
    "Israeli occupation forces slaughtered defenseless civilians in a savage massacre.", force=True)
show(result, trace)

INPUT: Israeli occupation forces slaughtered defenseless civilians in a savage massacre.
1 DETECTOR   status=biased  score=0.99  label=against_SIDE_A
2 EXPLAIN    flagged words: ['massacre', 'occupation', 'defenseless', 'civilians', 'savage']
3 RAG        query='massacre occupation defenseless civilians savage'  rules=['R01', 'R04', 'R18', 'R12']
4 REWRITE    round 1: Israeli forces killed civilians in an attack.
               "Israeli occupation forces slaughtered defenseless civilians in a savage massacre." -> "Israeli forces killed civilians in an attack."   [R01: actor not stated in the text]
               "massacre" -> "attack"   [R18: massacre, slaughter, mass murder]
               "savage" -> ""   [R04: dehumanizing language]
5 FACT CHECK ok=True  new numbers=[]  new names=[]
6 REVIEW     palestinian  fair=True  issues=[]
6 REVIEW     israeli      fair=True  issues=[]
7 EVALUATOR  approve
FINAL: Israeli forces killed civilians in an attack.
Changes made to this answer:
- "Isr

## 4. All test sentences

Uses Amen's `data/test_sentences.csv` when it has rows; otherwise a few built-in examples.
`score_after` re-scores the rewrite with the same detector. That number is circular (the rewriter edits the
words this detector flagged), so it is only a sanity check; the real measures are the second detector and
the human ratings.

In [ ]:
import pandas as pd

tests = pd.read_csv(config.TEST_SENTENCES_PATH) if config.TEST_SENTENCES_PATH.exists() else pd.DataFrame()
if tests.empty:
    print("test_sentences.csv is empty: using built-in examples")
    tests = pd.DataFrame({
        "id": [1, 2, 3, 4, 5, 6],
        "conflict": ["israel_palestine"] * 6,
        "biased": [
            "Hamas terrorists butchered innocent families in a barbaric massacre.",
            "Israeli occupation forces slaughtered defenseless civilians in a savage massacre.",
            "The Zionist regime bombed another hospital in Gaza today.",
            "Palestinian rioters attacked soldiers near Nablus, the army said.",
            "Talks between the two sides resumed in Cairo on Tuesday.",
            "At least 20 people were killed in a strike on a school in Gaza, according to local health officials.",
        ],
        "neutral": [""] * 6,
    })

rows = []
for _, t in tests.iterrows():
    conflict = t.get("conflict", config.DEFAULT_CONFLICT) or config.DEFAULT_CONFLICT
    result, trace = pipeline.run_detailed(t["biased"], conflict, force=True)
    d = trace["detect"]
    rows.append({
        "id": t["id"],
        "detector": f"{d['status']} {d['score']:.2f}",
        "flagged": ", ".join(trace.get("spans", [])),
        "rules": " ".join(rule_ids(trace)),
        "original": t["biased"],
        "revised": result["final"],
        "reference_neutral": t.get("neutral", ""),
        "n_changes": len(result["changes"]),
        "decision": result["decision"],
        "rounds": len(trace["rounds"]),
        "score_after": round(detector.detect(result["final"], conflict)["score"], 2),
        "seconds": trace["seconds"]["total"],
    })

demo = pd.DataFrame(rows)
demo.to_csv(ARTIFACTS / "pipeline_demo.csv", index=False)
pd.set_option("display.max_colwidth", 120)
demo

In [ ]:
print("decisions:", demo["decision"].value_counts().to_dict())
print("sentences changed:", int((demo["n_changes"] > 0).sum()), "of", len(demo))
print(f"seconds per sentence: mean {demo['seconds'].mean():.1f}, max {demo['seconds'].max():.1f}")
print("detector verdicts without force:", demo["detector"].str.split().str[0].value_counts().to_dict())

## What to look at

- **Flagged words**: framing words ("terrorists", "slaughtered"), not names? (Names are masked, so they should never appear.)
- **Rules**: R01 always first; do the other rules match the flagged words?
- **Rewrite**: only the flagged wording changed, facts and numbers identical, compared with Amen's `reference_neutral`?
- **Symmetry**: sentences 1 and 2 (built-in examples) get the same kind of fix.
- **Detector verdicts**: how many sentences land in "uncertain"? That decides how wide the human-review band can be.
- **Seconds per sentence**: needed to plan the Week 12 full runs (~200 probes, up to 3 rounds each).

Every run is also appended to `ARTIFACTS/audit_log.jsonl`.